# Phase 1: RAG Evaluation — The Vibe Check Fallacy


## 1. Imports and Configuration


In [ ]:
#@title Install dependencies
%pip install -q langchain langchain-community langchain-openai tiktoken
print('✓ Dependencies installed')


In [ ]:
#@title Configuration { display-mode: "form" }
import os
import pandas as pd
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

OPENAI_API_KEY = ""  #@param {type:"string"}
OPENAI_MODEL = "gpt-4o-mini"  #@param {type:"string"}

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print('✓ Configuration complete')
print('Model:', os.getenv('OPENAI_MODEL', OPENAI_MODEL))


## 2. Load Wikipedia Article


In [ ]:
urls = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

print("Loaded documents:", len(docs_list))
print(docs_list[0].page_content[:500])


## 3. Split Documents


In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250,
    chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

print("Number of chunks:", len(doc_splits))


## 4. Create Vector Store and Retriever


In [ ]:
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print('✓ Vector store and retriever ready')


## 5. Create LLM


In [ ]:
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", OPENAI_MODEL),
    temperature=0
)

print('✓ LLM ready:', llm.model_name)


## 6. Define RAG Function


In [ ]:
def rag_answer(question: str, system_prompt: str) -> dict:
    retrieved_docs = retriever.invoke(question)

    retrieved_context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "Context:\n{context}\n\nQuestion:\n{question}")
    ])

    chain = prompt | llm

    response = chain.invoke({
        "context": retrieved_context,
        "question": question
    })

    return {
        "question": question,
        "retrieved_context": retrieved_context,
        "generated_answer": response.content
    }

print('✓ RAG function defined')


## 7. Define 5 Questions


In [ ]:
questions = [
    "What is retrieval-augmented generation?",
    "Why does RAG retrieve external knowledge before generating an answer?",
    "What are the main components of a RAG system?",
    "How can RAG help reduce hallucinations?",
    "What are common applications of retrieval-augmented generation?"
]

print(f'✓ {len(questions)} questions defined')


## 8. Define Baseline System Prompt


In [ ]:
baseline_system_prompt = """
You are a careful RAG assistant.
Answer the question using only the provided context.
If the context does not contain enough information, say that the context is insufficient.
Do not guess or use outside knowledge.
Keep the answer concise and factual.
"""

print('✓ Baseline system prompt defined')


## 9. Run Baseline RAG


In [ ]:
baseline_results = [
    rag_answer(question, baseline_system_prompt)
    for question in questions
]

baseline_df = pd.DataFrame(baseline_results)
baseline_df


## 10. Manual Baseline Evaluation Table


In [ ]:
baseline_manual_eval_df = baseline_df.copy()
baseline_manual_eval_df["manual_quality_label"] = ""
baseline_manual_eval_df["manual_notes"] = ""

baseline_manual_eval_df[
    [
        "question",
        "retrieved_context",
        "generated_answer",
        "manual_quality_label",
        "manual_notes"
    ]
]


## 11. Define Degraded System Prompt


In [ ]:
degraded_system_prompt = """
You are a helpful assistant.
Answer the question as fully as possible.
You may use the provided context, but you can also rely on your general knowledge.
If the context is incomplete, make a reasonable guess.
Give a detailed answer.
"""

print('✓ Degraded system prompt defined')


## 12. Run Degraded RAG


In [ ]:
degraded_results = [
    rag_answer(question, degraded_system_prompt)
    for question in questions
]

degraded_df = pd.DataFrame(degraded_results)
degraded_df


## 13. Comparison Table


In [ ]:
comparison_df = pd.DataFrame({
    "question": baseline_df["question"],
    "baseline_answer": baseline_df["generated_answer"],
    "degraded_answer": degraded_df["generated_answer"],
    "baseline_context": baseline_df["retrieved_context"],
    "degraded_context": degraded_df["retrieved_context"],
    "manual_comparison_notes": ""
})

comparison_df


## 14. Export Results


In [ ]:
baseline_df.to_csv("phase1_baseline_results.csv", index=False)
degraded_df.to_csv("phase1_degraded_results.csv", index=False)
comparison_df.to_csv("phase1_comparison_results.csv", index=False)

print('✓ Exported:')
print('  phase1_baseline_results.csv')
print('  phase1_degraded_results.csv')
print('  phase1_comparison_results.csv')
